# RDS PostgreSQL Demo

Chỉ 4 bước: `create()` -> `status()` -> `seed()` -> `destroy(yes=True)`.

**Trước khi chạy**
- Mở notebook từ thư mục `database/RDS`.
- Chạy cell install bên dưới (chỉ cần lần đầu).
- `aws configure` và điền `RDS_MASTER_PASSWORD` trong `.env`.

> Port cố định **5432**. Mạng công ty thường chặn outbound 5432 — nếu `seed()` bị
> `Connection timed out` thì đổi sang hotspot 4G rồi chạy lại.

In [15]:
# %conda install -n base ipykernel --update-deps --force-reinstall
%pip install boto3 psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Setup 1 — Config & helpers

In [16]:
import os
import urllib.request

import boto3
from botocore.exceptions import ClientError

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.getcwd(), ".env"))
except ImportError:
    pass

SCRIPT_DIR = os.getcwd()
AWS_REGION = os.environ.get("AWS_REGION", "ap-southeast-1")
PROJECT = os.environ.get("PROJECT", "archival-demo")
RDS_ID = f"{PROJECT}-pg"
RDS_PORT = 5432
RDS_ENGINE_VERSION = os.environ.get("RDS_ENGINE_VERSION", "")
RDS_INSTANCE_CLASS = os.environ.get("RDS_INSTANCE_CLASS", "db.t4g.micro")
RDS_STORAGE_GB = int(os.environ.get("RDS_STORAGE_GB", "20"))
RDS_DB_NAME = os.environ.get("RDS_DB_NAME", "archivedemo")
RDS_MASTER_USER = os.environ.get("RDS_MASTER_USER", "demoadmin")
RDS_MASTER_PASSWORD = os.environ.get("RDS_MASTER_PASSWORD", "")
RDS_BACKUP_RETENTION_DAYS = int(os.environ.get("RDS_BACKUP_RETENTION_DAYS", "1"))
SG_NAME = f"{PROJECT}-rds-sg"
SUBNET_GROUP_NAME = f"{PROJECT}-rds-subnets"
OLD_ORDER_DAYS = 90


def my_ip():
    with urllib.request.urlopen("https://checkip.amazonaws.com", timeout=10) as r:
        return r.read().decode().strip()


def instance_exists(rds):
    try:
        rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "DBInstanceNotFound":
            return False
        raise


def find_sg(ec2):
    groups = ec2.describe_security_groups(
        Filters=[{"Name": "group-name", "Values": [SG_NAME]}]
    ).get("SecurityGroups", [])
    return groups[0]["GroupId"] if groups else None


def open_port(ec2, sg_id, cidr):
    """Mở 5432 cho cidr. Bỏ qua nếu rule đã có."""
    try:
        ec2.authorize_security_group_ingress(
            GroupId=sg_id,
            IpPermissions=[{
                "IpProtocol": "tcp", "FromPort": RDS_PORT, "ToPort": RDS_PORT,
                "IpRanges": [{"CidrIp": cidr}],
            }],
        )
        print(f"Opened {RDS_PORT} for {cidr}")
    except ClientError as e:
        if e.response["Error"]["Code"] != "InvalidPermission.Duplicate":
            raise


def endpoint():
    rds = boto3.client("rds", region_name=AWS_REGION)
    inst = rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)["DBInstances"][0]
    ep = inst.get("Endpoint") or {}
    if not ep.get("Address"):
        raise RuntimeError("Endpoint chưa sẵn sàng. Chạy status() tới khi available.")
    return ep["Address"], ep.get("Port", RDS_PORT)


def connect():
    """Tự mở firewall cho IP hiện tại (IP nhà mạng hay đổi) rồi kết nối."""
    import psycopg2
    ec2 = boto3.client("ec2", region_name=AWS_REGION)
    sg_id = find_sg(ec2)
    if sg_id:
        open_port(ec2, sg_id, f"{my_ip()}/32")
    host, port = endpoint()
    return psycopg2.connect(
        host=host, port=port, user=RDS_MASTER_USER,
        password=RDS_MASTER_PASSWORD, dbname=RDS_DB_NAME, connect_timeout=15,
    )


def load_sql(name):
    with open(os.path.join(SCRIPT_DIR, name), "r", encoding="utf-8") as f:
        return f.read()


print(f"setup ready: region={AWS_REGION} project={PROJECT} port={RDS_PORT}")

setup ready: region=ap-southeast-1 project=archival-demo port=5432


## Setup 2 — create / status / seed / destroy

In [17]:
def _subnet_group(ec2, rds, vpc_id):
    try:
        rds.describe_db_subnet_groups(DBSubnetGroupName=SUBNET_GROUP_NAME)
        return SUBNET_GROUP_NAME
    except ClientError as e:
        if e.response["Error"]["Code"] != "DBSubnetGroupNotFoundFault":
            raise

    by_az = {}
    for s in ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpc_id]}])["Subnets"]:
        by_az.setdefault(s["AvailabilityZone"], s["SubnetId"])
    if len(by_az) < 2:
        zones = ec2.describe_availability_zones(
            Filters=[{"Name": "state", "Values": ["available"]}]
        )["AvailabilityZones"]
        for z in zones:
            az = z["ZoneName"]
            if az in by_az:
                continue
            try:
                by_az[az] = ec2.create_default_subnet(AvailabilityZone=az)["Subnet"]["SubnetId"]
            except ClientError:
                pass
            if len(by_az) >= 2:
                break
    if len(by_az) < 2:
        raise RuntimeError(f"RDS cần subnet ở 2 AZ trong VPC {vpc_id}")

    rds.create_db_subnet_group(
        DBSubnetGroupName=SUBNET_GROUP_NAME,
        DBSubnetGroupDescription=f"Dev/test RDS subnets for {PROJECT}",
        SubnetIds=list(by_az.values()),
        Tags=[{"Key": "Project", "Value": PROJECT}],
    )
    print(f"Created subnet group {SUBNET_GROUP_NAME} ({', '.join(by_az)})")
    return SUBNET_GROUP_NAME


def _engine_version(rds):
    if RDS_ENGINE_VERSION:
        print("RDS_ENGINE_VERSION =", repr(RDS_ENGINE_VERSION))
        print("Type =", type(RDS_ENGINE_VERSION))
        print("Resolved =", repr(_engine_version(rds)))
        return RDS_ENGINE_VERSION
    versions = []
    for page in rds.get_paginator("describe_db_engine_versions").paginate(Engine="postgres"):
        versions += [v["EngineVersion"] for v in page["DBEngineVersions"]]
    picked = max(versions, key=lambda v: tuple(int(p) for p in v.split(".") if p.isdigit()))
    print(f"PostgreSQL version: {picked}")
    return picked


def create():
    if not RDS_MASTER_PASSWORD:
        raise RuntimeError("Thiếu RDS_MASTER_PASSWORD trong .env")
    session = boto3.Session(region_name=AWS_REGION)
    ec2, rds = session.client("ec2"), session.client("rds")
    account = session.client("sts").get_caller_identity()["Account"]
    print(f"AWS account {account} / {AWS_REGION}")

    vpcs = ec2.describe_vpcs(Filters=[{"Name": "isDefault", "Values": ["true"]}])["Vpcs"]
    if not vpcs:
        raise RuntimeError(f"Không có default VPC ở {AWS_REGION}")
    vpc_id = vpcs[0]["VpcId"]
    print(f"Default VPC: {vpc_id}")

    subnet_group = _subnet_group(ec2, rds, vpc_id)
    sg_id = find_sg(ec2)
    if not sg_id:
        sg_id = ec2.create_security_group(
            GroupName=SG_NAME,
            Description=f"Dev/test RDS access for {PROJECT}",
            VpcId=vpc_id,
        )["GroupId"]
        print(f"Created security group {sg_id}")
    open_port(ec2, sg_id, f"{my_ip()}/32")

    if instance_exists(rds):
        print(f"{RDS_ID} đã tồn tại, bỏ qua create.")
        return
    rds.create_db_instance(
        DBInstanceIdentifier=RDS_ID,
        Engine="postgres",
        EngineVersion=_engine_version(rds),
        DBInstanceClass=RDS_INSTANCE_CLASS,
        AllocatedStorage=RDS_STORAGE_GB,
        StorageType="gp3",
        DBName=RDS_DB_NAME,
        MasterUsername=RDS_MASTER_USER,
        MasterUserPassword=RDS_MASTER_PASSWORD,
        Port=RDS_PORT,
        DBSubnetGroupName=subnet_group,
        VpcSecurityGroupIds=[sg_id],
        BackupRetentionPeriod=RDS_BACKUP_RETENTION_DAYS,
        MultiAZ=False,
        PubliclyAccessible=True,
        EnablePerformanceInsights=False,
        AutoMinorVersionUpgrade=False,
        Tags=[{"Key": "Project", "Value": PROJECT}, {"Key": "Environment", "Value": "dev"}],
    )

    print("Đang tạo RDS, vui lòng chờ...")

    waiter = rds.get_waiter("db_instance_available")
    waiter.wait(
        DBInstanceIdentifier=RDS_ID,
        WaiterConfig={
            "Delay": 30,
            "MaxAttempts": 40,
        },
    )

    print(f"✓ RDS {RDS_ID} đã AVAILABLE.")

def status():
    rds = boto3.client("rds", region_name=AWS_REGION)

    if not instance_exists(rds):
        print(f"{RDS_ID} không tồn tại. Chạy create() trước.")
        return

    inst = rds.describe_db_instances(
        DBInstanceIdentifier=RDS_ID
    )["DBInstances"][0]

    ep = inst.get("Endpoint") or {}

    # Lấy subnet và security group
    subnet_group = inst["DBSubnetGroup"]["DBSubnetGroupName"]
    subnet_ids = [s["SubnetIdentifier"] for s in inst["DBSubnetGroup"]["Subnets"]]
    sg_ids = [sg["VpcSecurityGroupId"] for sg in inst["VpcSecurityGroups"]]

    print(f"Status       : {inst['DBInstanceStatus']}")
    print(f"RDS ID       : {inst['DBInstanceIdentifier']}")
    print(f"Subnet Group : {subnet_group}")
    print(f"Subnet IDs   : {', '.join(subnet_ids)}")
    print(f"Security GPs : {', '.join(sg_ids)}")

    if not ep.get("Address"):
        print("\nEndpoint chưa sẵn sàng — chạy lại status() sau vài phút.")
        return

    host, port = ep["Address"], ep.get("Port", RDS_PORT)

    print("\n--- Kết nối bằng pgAdmin / DBeaver ---")
    print(f"Host         : {host}")
    print(f"Port         : {port}")
    print(f"Database     : {RDS_DB_NAME}")
    print(f"Username     : {RDS_MASTER_USER}")
    print(f"Password     : {RDS_MASTER_PASSWORD}")
    print("SSL mode     : prefer")

    print("\nURI:")
    print(
        f"postgresql://{RDS_MASTER_USER}:{RDS_MASTER_PASSWORD}"
        f"@{host}:{port}/{RDS_DB_NAME}"
    )

def seed():
    raw = load_sql("seed.sql")
    body = "\n".join(l for l in raw.splitlines() if not l.strip().startswith("\\set"))
    body = body.replace(":old_order_days", str(OLD_ORDER_DAYS))

    conn = connect()
    conn.autocommit = True
    try:
        with conn.cursor() as cur:
            print("Applying schema.sql ...")
            cur.execute(load_sql("schema.sql"))
            print("Applying seed.sql ...")
            cur.execute(body)
            if cur.description:
                cols = [d.name for d in cur.description]
                row = cur.fetchone()
                if row:
                    print("Summary: " + ", ".join(f"{c}={v}" for c, v in zip(cols, row)))
            for t in ("customers", "orders", "order_items"):
                cur.execute(f"SELECT count(*) FROM {t}")
                print(f"{t}: {cur.fetchone()[0]} rows")
    finally:
        conn.close()
    print("Seed xong.")


def destroy(yes=False):
    if not yes:
        print("Cần gọi destroy(yes=True) để xác nhận.")
        return
    session = boto3.Session(region_name=AWS_REGION)
    rds, ec2 = session.client("rds"), session.client("ec2")

    if instance_exists(rds):
        print(f"Deleting {RDS_ID} ...")
        rds.delete_db_instance(
            DBInstanceIdentifier=RDS_ID, SkipFinalSnapshot=True, DeleteAutomatedBackups=True
        )
        print("Waiting for deletion ...")
        rds.get_waiter("db_instance_deleted").wait(DBInstanceIdentifier=RDS_ID)
        print("Instance deleted.")
    else:
        print(f"{RDS_ID} không tồn tại, bỏ qua.")

    sg_id = find_sg(ec2)
    if sg_id:
        try:
            ec2.delete_security_group(GroupId=sg_id)
            print(f"Deleted security group {sg_id}")
        except ClientError:
            print("Chưa xóa được security group, thử lại sau ít phút.")
    try:
        rds.delete_db_subnet_group(DBSubnetGroupName=SUBNET_GROUP_NAME)
        print(f"Deleted subnet group {SUBNET_GROUP_NAME}")
    except ClientError:
        pass
    print("Teardown xong.")


print("commands ready: create() / status() / seed() / destroy(yes=True)")

commands ready: create() / status() / seed() / destroy(yes=True)


## Bước 1 — Tạo RDS instance
Mất vài phút.

In [ ]:
create()

AWS account 637423316258 / ap-southeast-1
Default VPC: vpc-0701a5b03f066537c
Created subnet group archival-demo-rds-subnets (ap-southeast-1b, ap-southeast-1a)
PostgreSQL version: 18.6
Đang tạo RDS, vui lòng chờ...


## Bước 2 — Trạng thái + thông tin kết nối
Chạy lại tới khi thấy `available`.

In [ ]:
status()

## Bước 3 — Nạp schema + fake data

In [ ]:
seed()

## Bước 4 — Xóa toàn bộ tài nguyên
RDS không free khi chạy, nhớ chạy cell này khi xong.

In [ ]:
# destroy(yes=True)